# DNN Deep SVDD — Compute Selective p-value

Load a trained MLP encoder → generate test & reference data →
build a `pythonsi` Pipeline with `DeepSVDDAD(network_type="dnn")` → compute
selective p-values.

**Prerequisite:** run `train.ipynb` first to produce `weights/mlp_encoder.pth`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch

REPO = Path.cwd()
while REPO.name and not (REPO / "pythonsi").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from pythonsi import Data, Pipeline
from pythonsi.anomaly_detection import DeepSVDDAD
from pythonsi.test_statistics import DeepSVDDTestStatistic
from network import MLP

In [2]:
SEED = 7
N_FEATURES = 20
N_REFS = 20
ALPHA = 0.05

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Load Trained Model

In [3]:
ckpt = torch.load("weights/mlp_encoder.pth", map_location=DEVICE, weights_only=False)
cfg = ckpt["config"]

model = MLP(
    n_features=cfg["n_features"],
    hidden_dim=cfg["hidden_dim"],
    repdim=cfg["repdim"],
).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

center_c = ckpt["center_c"]
R_squared = ckpt["R_squared"]
print(f"R² = {R_squared:.10f}")

R² = 0.0097502451


## 2. Generate Test & Reference Data

In [4]:
rng = np.random.default_rng(SEED)

# Reference data: normal N(0,1)
X_refs = rng.normal(size=(N_REFS, N_FEATURES)).astype(np.float64)

# Test data: anomalous (shifted mean)
X_test = rng.normal(size=(1, N_FEATURES)).astype(np.float64)
X_test[0] += 3.0

# Check if test triggers anomaly
with torch.no_grad():
    feat = model(torch.from_numpy(X_test).float().to(DEVICE))
    score = torch.sum((feat - torch.tensor(center_c, device=DEVICE)) ** 2).item()

print(f"Score:  {score:.10f}")
print(f"R²:     {R_squared:.10f}")
print(f"Anomaly detected: {score > R_squared}")

Score:  0.5670268536
R²:     0.0097502451
Anomaly detected: True


## 3. Build Pipeline & Compute Selective p-value

In [ ]:
test_node = Data()
refs_node = Data()

detector = DeepSVDDAD(
    model=model,
    R_squared=R_squared,
    center=center_c,
    device="cpu",
    network_type="dnn",
)
anomaly_node = detector.run(test_node)

pipeline = Pipeline(
    inputs=(test_node, refs_node),
    output=anomaly_node,
    test_statistic=DeepSVDDTestStatistic(test_node, refs_node),
)

print("Running selective inference …")
anomalies, p_values = pipeline(
    inputs=[X_test, X_refs],
    covariances=[np.eye(N_FEATURES)],
    verbose=False,
)

print(f"\nDetected anomalies: {anomalies}")
print(f"Selective p-values: {p_values}")

if len(p_values) > 0 and p_values[0] is not None:
    print(f"Reject H₀ at α={ALPHA}? {'YES' if p_values[0] < ALPHA else 'NO'}")

Running selective inference …
Selected output: [0]
Testing feature 0
Feature 0: p-value = 0.0

Detected anomalies: [0]
Selective p-values: [0.0]
Reject H₀ at α=0.05? YES
